# ※ 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.

- 채점 기준:
    - 출력 결과 일치: 제출한 코드가 제시된 출력 결과와 일치하는 경우에만 정답으로 인정됩니다.
    - 코드의 다양성 인정: 출력 결과가 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# VectorDB 과제

## 공통 설치 라이브러리

In [ ]:
# !pip install sentence-transformers openai chromadb rank-bm25 scikit-learn httpx fastapi uvicorn pydantic numpy python-dotenv

In [ ]:
import os
import numpy as np
import time
import chromadb
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# .env 파일에서 API 키 로드
load_dotenv()

print("환경 설정 완료!")

---
## 문제 1. 멀티 모델 임베딩 비교 벤치마크 구현 (10점)

두 가지 임베딩 모델(오픈소스 SentenceTransformer vs OpenAI API)의 한국어 문장 쌍 유사도 성능을 비교하세요.

- 코사인 유사도 정확도, 속도를 측정합니다.
- accuracy, precision, recall, 유사/비유사 평균 유사도를 계산합니다.

In [ ]:
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import numpy as np
import time

# 평가 데이터셋: (문장A, 문장B, 유사 여부 True/False)
EVAL_PAIRS = [
    ("인공지능이 의료 분야를 혁신하고 있다.", "AI 기술이 병원에서 활용되고 있다.", True),
    ("오늘 서울 날씨가 맑습니다.", "서울의 하늘이 화창하고 좋다.", True),
    ("주식 시장이 크게 하락했다.", "금융 위기가 발생했다.", True),
    ("인공지능이 의료 분야를 혁신하고 있다.", "오늘 저녁에 파스타를 만들 예정이다.", False),
    ("서울에서 부산까지 KTX로 2시간 걸린다.", "고양이가 소파에서 낮잠을 자고 있다.", False),
    ("딥러닝 모델의 학습률을 조정했다.", "맛있는 김치찌개 레시피를 알려주세요.", False),
    ("클라우드 서버 비용이 증가하고 있다.", "AWS 요금이 올랐다.", True),
    ("자연어 처리 기술이 발전하고 있다.", "NLP 분야에서 새로운 모델이 등장했다.", True),
    ("오늘 점심에 비빔밥을 먹었다.", "내일 회의가 오후 3시에 있다.", False),
    ("파이썬은 데이터 분석에 널리 사용된다.", "Python은 ML/AI에서 인기있는 언어이다.", True),
]

class EmbeddingBenchmark:
    def __init__(self):
        # TODO: 오픈소스 모델 로드 (paraphrase-multilingual-MiniLM-L12-v2)
        self.local_model = None
        self.openai_client = OpenAI()
    
    def embed_local(self, texts):
        """오픈소스 모델로 임베딩 생성"""
        # TODO: SentenceTransformer로 임베딩 생성, numpy array 반환
        pass
    
    def embed_api(self, texts, model="text-embedding-3-small"):
        """OpenAI API로 임베딩 생성"""
        # TODO: OpenAI Embeddings API 호출, numpy array 반환
        pass
    
    def cosine_similarity(self, a, b):
        """코사인 유사도 계산"""
        # TODO: numpy로 코사인 유사도 계산
        pass
    
    def evaluate(self, embed_func, threshold=0.5):
        """
        임베딩 함수의 성능 평가
        
        Returns:
            dict: {"accuracy": float, "precision": float, "recall": float, 
                   "avg_sim_positive": float, "avg_sim_negative": float, "elapsed_ms": float}
        """
        start = time.time()
        
        # TODO: 모든 문장 쌍에 대해 임베딩 생성 및 유사도 계산
        # TODO: threshold 기준으로 예측 (유사도 >= threshold → True)
        # TODO: accuracy, precision, recall 계산
        # TODO: 유사 쌍 / 비유사 쌍 평균 유사도 각각 계산
        
        elapsed = (time.time() - start) * 1000
        
        return {
            "accuracy": None,      # TODO
            "precision": None,     # TODO
            "recall": None,        # TODO
            "avg_sim_positive": None,  # TODO
            "avg_sim_negative": None,  # TODO
            "elapsed_ms": elapsed
        }

bench = EmbeddingBenchmark()
local_result = bench.evaluate(bench.embed_local)
api_result = bench.evaluate(bench.embed_api)

print("=" * 65)
print(f"{'\uc9c0\ud45c':>20} | {'\uc624\ud508\uc18c\uc2a4 (MiniLM)':>16} | {'API (OpenAI)':>16}")
print("-" * 65)
for key in ["accuracy", "precision", "recall", "avg_sim_positive", "avg_sim_negative", "elapsed_ms"]:
    lv = local_result[key]
    av = api_result[key]
    fmt = ".1f" if key == "elapsed_ms" else ".4f"
    print(f"{key:>20} | {lv:>16{fmt}} | {av:>16{fmt}}")

---
## 문제 2. 임베딩 차원 축소 + 시각화 분석 (10점)

PCA로 임베딩을 2D로 축소하고, 클러스터 품질을 실루엿 스코어로 평가하세요.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

DOCUMENTS = [
    # 카테고리 A: AI/기술
    ("딥러닝 모델의 학습률을 조정하는 방법", "tech"),
    ("자연어 처리에서 트랜스포머 아키텍처의 역할", "tech"),
    ("GPU를 활용한 대규모 모델 학습 최적화", "tech"),
    ("강화학습 에이전트의 보상 함수 설계", "tech"),
    # 카테고리 B: 요리/음식
    ("김치찌개를 맛있게 끌이는 비법", "food"),
    ("이탈리안 파스타의 정통 레시피", "food"),
    ("건강한 샐러드 드레싱 만들기", "food"),
    ("한국 전통 떡 만드는 방법", "food"),
    # 카테고리 C: 여행
    ("제주도 3박 4일 여행 코스 추천", "travel"),
    ("유럽 배낙여행 필수 준비물 리스트", "travel"),
    ("도쿄 맛집 투어 가이드", "travel"),
    ("발리 서핑 스팟 베스트 5", "travel"),
]

texts = [d[0] for d in DOCUMENTS]
labels = [d[1] for d in DOCUMENTS]

# TODO: SentenceTransformer로 임베딩 생성
embeddings = None

# TODO: PCA로 2차원 축소
pca = None
reduced = None

# TODO: 실루엿 스코어 계산 (클러스터 품질)
sil_score = None

print(f"실루엿 스코어: {sil_score:.4f}")
print(f"(1에 가까울수록 클러스터가 잘 분리됨)\n")

# TODO: 카테고리별 2D 중심점(centroid) 계산
centroids = {}
for label in set(labels):
    # TODO: 해당 카테고리의 2D 좌표 평균
    pass

# TODO: 카테고리 간 중심점 거리 계산
print("카테고리 간 중심점 거리:")
for l1 in sorted(centroids.keys()):
    for l2 in sorted(centroids.keys()):
        if l1 < l2:
            dist = np.linalg.norm(centroids[l1] - centroids[l2])
            print(f"  {l1} \u2194 {l2}: {dist:.4f}")

---
## 문제 3. ChromaDB CRUD + 메타데이터 필터링 시스템 (10점)

ChromaDB를 활용한 완전한 문서 관리 시스템을 구축하세요. CRUD 연산과 복합 메타데이터 필터를 구현합니다.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

TECH_DOCS = [
    {"id": "doc_001", "text": "쿠버네티스 파드(Pod)는 컨테이너의 최소 배포 단위입니다.", "category": "kubernetes", "difficulty": "beginner", "year": 2024},
    {"id": "doc_002", "text": "도커 이미지는 Dockerfile로부터 빌드됩니다.", "category": "docker", "difficulty": "beginner", "year": 2024},
    {"id": "doc_003", "text": "Redis는 인메모리 데이터 저장소로 캐싱에 사용됩니다.", "category": "database", "difficulty": "intermediate", "year": 2023},
    {"id": "doc_004", "text": "PostgreSQL의 EXPLAIN ANALYZE는 쿼리 실행 계획을 보여줍니다.", "category": "database", "difficulty": "advanced", "year": 2023},
    {"id": "doc_005", "text": "Kafka는 분산 이벤트 스트리밍 플랫폼입니다.", "category": "streaming", "difficulty": "intermediate", "year": 2024},
    {"id": "doc_006", "text": "Helm 차트는 쿠버네티스 애플리케이션 패키징 도구입니다.", "category": "kubernetes", "difficulty": "intermediate", "year": 2024},
    {"id": "doc_007", "text": "Prometheus는 메트릭 수집 및 모니터링 시스템입니다.", "category": "monitoring", "difficulty": "intermediate", "year": 2023},
    {"id": "doc_008", "text": "Terraform은 인프라를 코드로 관리하는 IaC 도구입니다.", "category": "iac", "difficulty": "intermediate", "year": 2024},
    {"id": "doc_009", "text": "gRPC는 고성능 RPC 프레임워크로 마이크로서비스에 적합합니다.", "category": "networking", "difficulty": "advanced", "year": 2023},
    {"id": "doc_010", "text": "GitHub Actions는 CI/CD 파이프라인 자동화 도구입니다.", "category": "cicd", "difficulty": "beginner", "year": 2024},
    {"id": "doc_011", "text": "HPA(Horizontal Pod Autoscaler)는 CPU 사용률 기반으로 파드를 자동 스케일링합니다.", "category": "kubernetes", "difficulty": "advanced", "year": 2024},
    {"id": "doc_012", "text": "ArgoCD는 GitOps 기반 쿠버네티스 배포 도구입니다.", "category": "cicd", "difficulty": "advanced", "year": 2024},
]

model = SentenceTransformer('all-MiniLM-L6-v2')

# TODO: ChromaDB 클라이언트 생성 (인메모리)
client = None

# TODO: 컨렉션 생성 (cosine distance 사용)
collection = None

# TODO: 모든 문서를 임베딩과 함께 적재
# (ids, documents, embeddings, metadatas 모두 포함)

print(f"적재된 문서 수: {collection.count()}")

# === 검색 테스트 ===

# 검색 1: 기본 시맨틱 검색
query1 = "컨테이너 오케스트레이션 도구"
# TODO: 시맨틱 검색 (top 3)
results1 = None
print(f"\n[검색 1] '{query1}'")
for i, (doc, score) in enumerate(zip(results1['documents'][0], results1['distances'][0])):
    print(f"  {i+1}. (거리: {score:.4f}) {doc[:60]}...")

# 검색 2: 메타데이터 필터 (category == "kubernetes" AND difficulty != "beginner")
# TODO: where 필터 적용 검색
results2 = None
print(f"\n[검색 2] kubernetes + intermediate/advanced:")
for doc, meta in zip(results2['documents'][0], results2['metadatas'][0]):
    print(f"  [{meta['difficulty']}] {doc[:60]}...")

# 검색 3: 복합 필터 (year >= 2024 AND difficulty in ["beginner", "intermediate"])
# TODO: $and 복합 필터 적용
results3 = None
print(f"\n[검색 3] 2024년 + 초급/중급:")
for doc, meta in zip(results3['documents'][0], results3['metadatas'][0]):
    print(f"  [{meta['category']}/{meta['difficulty']}] {doc[:60]}...")

# === 업데이트 & 삭제 ===
# TODO: doc_003의 difficulty를 "advanced"로 업데이트
# TODO: doc_009 삭제
# TODO: 최종 문서 수 확인
print(f"\n최종 문서 수: {collection.count()}")

---
## 문제 4. ETL 파이프라인 구축: 텍스트 → 청킹 → 임베딩 → ChromaDB (10점)

슬라이딩 윈도우 청킹, 키워드 추출, 메타데이터 생성 후 ChromaDB에 적재하는 전체 ETL 파이프라인을 구현하세요.

In [ ]:
RAW_DOCUMENTS = [
    {
        "title": "쿠버네티스 입문 가이드",
        "content": """쿠버네티스(Kubernetes, K8s)는 컨테이너화된 워크로드와 서비스를 관리하기 위한 이식 가능하고 확장 가능한 오픈소스 플랫폼입니다. 쿠버네티스는 선언적 구성과 자동화를 모두 지원합니다. 파드(Pod)는 쿠버네티스에서 생성하고 관리할 수 있는 배포 가능한 가장 작은 컴퓨팅 단위입니다. 파드는 하나 이상의 컨테이너 그룹으로, 스토리지와 네트워크를 공유하며 컨테이너를 실행하는 방법에 대한 명세를 갖습니다. 디플로이먼트(Deployment)는 파드와 레플리카셋에 대한 선언적 업데이트를 제공합니다. 서비스(Service)는 파드 집합에서 실행되는 애플리케이션을 네트워크 서비스로 노출하는 추상화된 방법입니다. 인그레스(Ingress)는 클러스터 외부에서 내부 서비스로의 HTTP/HTTPS 라우팅 규칙을 관리합니다.""",
        "source": "k8s-docs",
    },
    {
        "title": "FastAPI 프레임워크 소개",
        "content": """FastAPI는 파이썬 3.7+를 기반으로 하는 현대적이고 빠른 웹 프레임워크입니다. 주요 특징은 자동 API 문서화(Swagger UI, ReDoc), 타입 힌트 기반의 데이터 검증, 비동기(async/await) 지원입니다. Pydantic 모델을 사용하여 요청과 응답의 스키마를 정의하면 자동으로 유효성 검사가 수행됩니다. 미들웨어를 통해 CORS, 인증, 레이트 리미팅 등을 구현할 수 있습니다. 의존성 주입(Dependency Injection) 시스템으로 코드의 재사용성과 테스트 용이성을 높입니다. 배포는 uvicorn이나 gunicorn과 함께 사용하며, Docker 컨테이너로 패키징하는 것이 일반적입니다.""",
        "source": "fastapi-docs",
    },
    {
        "title": "Redis 캐싱 전략",
        "content": """Redis는 인메모리 데이터 구조 저장소로, 캐시, 메시지 브로커, 세션 저장소 등으로 활용됩니다. 주요 캐싱 패턴으로는 Cache-Aside(Lazy Loading), Write-Through, Write-Behind가 있습니다. Cache-Aside는 애플리케이션이 먼저 캐시를 조회하고, 없으면 DB에서 읽어온 후 캐시에 저장하는 방식입니다. TTL(Time To Live)을 설정하여 캐시 만료를 관리합니다. Redis의 데이터 구조로는 String, Hash, List, Set, Sorted Set, Stream 등이 있습니다. Redis Cluster를 사용하면 데이터를 여러 노드에 분산하여 고가용성과 확장성을 확보할 수 있습니다.""",
        "source": "redis-docs",
    },
]

def sliding_window_chunk(text, chunk_size=150, overlap=30):
    """슬라이딩 윈도우 기반 텍스트 청킹 (문자 단위)"""
    # TODO: chunk_size 간격으로 텍스트 분할, overlap만큼 겹침
    # 반환: list of str
    pass

def extract_keywords(text, top_n=5):
    """TF 기반 키워드 추출 (단어 빈도 상위 N개)"""
    # TODO: 텍스트를 단어로 분리
    # TODO: 2글자 이상 단어만 필터링
    # TODO: 빈도 상위 top_n개 반환
    pass

def build_chunk_metadata(chunk_text, doc_title, doc_source, chunk_index):
    """청크별 메타데이터 생성"""
    # TODO: 키워드 추출 포함
    return {
        "title": doc_title,
        "source": doc_source,
        "chunk_index": chunk_index,
        "char_count": len(chunk_text),
        "keywords": None,  # TODO: extract_keywords 결과를 쉼표로 join
    }

def run_etl_pipeline(documents, collection_name="etl_docs"):
    """
    전체 ETL 파이프라인 실행
    
    Extract → Transform(청킹 + 메타데이터) → Load(ChromaDB)
    """
    client = chromadb.Client()
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # TODO: 컨렉션 생성 (기존 있으면 삭제 후 재생성)
    collection = None
    
    all_chunks = []
    all_ids = []
    all_metadatas = []
    
    for doc in documents:
        # Extract
        title = doc["title"]
        content = doc["content"]
        source = doc["source"]
        
        # Transform: 청킹
        # TODO: sliding_window_chunk 호출
        chunks = None
        
        for i, chunk in enumerate(chunks):
            chunk_id = f"{source}_chunk_{i:03d}"
            # TODO: 메타데이터 생성
            metadata = None
            
            all_chunks.append(chunk)
            all_ids.append(chunk_id)
            all_metadatas.append(metadata)
    
    # Load: 임베딩 생성 + ChromaDB 적재
    # TODO: 전체 청크 임베딩 생성
    embeddings = None
    
    # TODO: ChromaDB에 적재
    
    return collection, len(all_chunks)

collection, total = run_etl_pipeline(RAW_DOCUMENTS)
print(f"\u2705 ETL 완료: {total}개 청크 적재")

# 검증: 검색 테스트
test_queries = ["컨테이너 배포 방법", "캐시 만료 관리", "API 문서 자동화"]
for q in test_queries:
    results = collection.query(query_texts=[q], n_results=2)
    print(f"\n\U0001f50e '{q}':")
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        print(f"  [{meta['source']}] {doc[:60]}...")

---
## 문제 5. BM25 + Vector 하이브리드 검색 엔진 (10점)

BM25 키워드 검색과 벡터 시맨틱 검색을 Reciprocal Rank Fusion으로 융합하는 하이브리드 검색 엔진을 구현하세요.

In [ ]:
from rank_bm25 import BM25Okapi

# 검색 대상 문서 (ETL에서 적재한 컨렉션 사용)
SEARCH_CORPUS = [
    "쿠버네티스 파드는 컨테이너의 최소 배포 단위입니다.",
    "도커 이미지는 Dockerfile로부터 레이어 단위로 빌드됩니다.",
    "Redis TTL을 설정하여 캐시 만료를 자동 관리합니다.",
    "FastAPI는 async/await 기반의 비동기 웹 프레임워크입니다.",
    "Kafka 토픽은 파티션으로 나뉘어 병렬 처리됩니다.",
    "PostgreSQL JSONB 타입은 JSON 데이터를 효율적으로 저장합니다.",
    "Prometheus AlertManager는 메트릭 임계값 초과 시 알림을 발송합니다.",
    "Terraform state 파일은 인프라의 현재 상태를 추적합니다.",
    "Nginx 리버스 프록시는 로드 밸런싱과 SSL 종료를 처리합니다.",
    "GitHub Actions workflow는 YAML 파일로 CI/CD를 정의합니다.",
    "gRPC는 Protocol Buffers를 사용한 고성능 RPC 프레임워크입니다.",
    "Elasticsearch는 역인덱스 기반의 풀텍스트 검색 엔진입니다.",
    "ArgoCD는 GitOps 방식으로 쿠버네티스 배포를 자동화합니다.",
    "Celery는 분산 태스크 큐로 비동기 작업 처리에 사용됩니다.",
    "Grafana 대시보드는 다양한 데이터소스의 메트릭을 시각화합니다.",
]

model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()

# TODO: BM25 인덱스 생성 (한국어 토큰화: 공백 분리)
tokenized_corpus = None
bm25 = None

# TODO: ChromaDB 컨렉션 생성 및 문서 적재 (임베딩 포함)
hybrid_collection = None

def search_bm25(query, top_k=5):
    """BM25 검색"""
    # TODO: 쿼리 토큰화 → BM25 점수 계산 → 상위 K개 반환
    # 반환: list of (index, score, document)
    pass

def search_vector(query, top_k=5):
    """벡터 검색 (ChromaDB)"""
    # TODO: ChromaDB 쿼리 → 상위 K개 반환
    # 반환: list of (index, score, document)  (score = 1 - distance)
    pass

def reciprocal_rank_fusion(bm25_results, vector_results, k=60):
    """
    Reciprocal Rank Fusion으로 두 검색 결과 융합
    RRF score = \u03a3 1/(k + rank_i) for each result list
    """
    # TODO: 각 문서의 BM25 순위와 Vector 순위에서 RRF 점수 계산
    # TODO: RRF 점수 기준 내림차순 정렬
    # 반환: list of (index, rrf_score, document)
    pass

def hybrid_search(query, top_k=5, mode="hybrid"):
    """통합 검색 인터페이스"""
    if mode == "bm25":
        return search_bm25(query, top_k)
    elif mode == "vector":
        return search_vector(query, top_k)
    elif mode == "hybrid":
        bm25_r = search_bm25(query, top_k * 2)
        vec_r = search_vector(query, top_k * 2)
        # TODO: RRF 융합 후 상위 top_k 반환
        pass
    else:
        raise ValueError(f"Unknown mode: {mode}")

# 테스트
query = "컨테이너 배포 자동화"
for mode in ["bm25", "vector", "hybrid"]:
    results = hybrid_search(query, top_k=3, mode=mode)
    print(f"\n[{mode.upper()}] '{query}':")
    for idx, score, doc in results:
        print(f"  (score={score:.4f}) {doc[:60]}...")

---
## 문제 6. Hit Rate & MRR 평가 프레임워크 (10점)

검색 시스템의 품질을 Hit Rate@K와 MRR@K로 평가하는 프레임워크를 구현하세요.

In [ ]:
# 평가 데이터셋: (query, relevant_doc_indices)
EVAL_DATASET = [
    ("쿠버네티스 배포 방법", [0, 12]),            # 파드, ArgoCD
    ("캐시 관리 전략", [2]),                       # Redis TTL
    ("CI/CD 파이프라인 구성", [9, 12]),            # GitHub Actions, ArgoCD
    ("비동기 웹 서버 프레임워크", [3, 13]),        # FastAPI, Celery
    ("메시지 큐 시스템", [4, 13]),                 # Kafka, Celery
    ("데이터베이스 쿼리 최적화", [5]),              # PostgreSQL JSONB
    ("인프라 모니터링 도구", [6, 14]),             # Prometheus, Grafana
    ("IaC 도구 비교", [7]),                        # Terraform
    ("컨테이너 이미지 빌드", [1]),                 # 도커
    ("서비스 간 통신 프로토콜", [10, 8]),          # gRPC, Nginx
]

def hit_rate_at_k(retrieved_indices, relevant_indices, k):
    """
    Hit Rate@K: 상위 K개 결과 중 관련 문서가 하나라도 있으면 1, 아니면 0
    """
    # TODO
    pass

def mrr_at_k(retrieved_indices, relevant_indices, k):
    """
    MRR@K (Mean Reciprocal Rank): 첫 번째 관련 문서의 역순위
    관련 문서가 없으면 0
    """
    # TODO
    pass

def evaluate_search_system(search_func, eval_dataset, ks=[1, 3, 5]):
    """
    검색 시스템 전체 평가
    
    Returns:
        dict: {k: {"hit_rate": float, "mrr": float}} for each k
    """
    results = {}
    
    for k in ks:
        hit_rates = []
        mrrs = []
        
        for query, relevant_ids in eval_dataset:
            # TODO: 검색 실행하여 상위 k개 문서 인덱스 추출
            retrieved = None
            
            # TODO: hit_rate, mrr 계산
            hr = None
            mr = None
            
            hit_rates.append(hr)
            mrrs.append(mr)
        
        results[k] = {
            "hit_rate": np.mean(hit_rates),
            "mrr": np.mean(mrrs)
        }
    
    return results

# 3가지 검색 방식 비교 평가
print("=" * 60)
print(f"{'\ubc29\uc2dd':>10} | {'K':>3} | {'Hit Rate':>10} | {'MRR':>10}")
print("-" * 60)

for mode in ["bm25", "vector", "hybrid"]:
    search_fn = lambda q, k, m=mode: [idx for idx, _, _ in hybrid_search(q, k, m)]
    eval_result = evaluate_search_system(search_fn, EVAL_DATASET)
    for k, metrics in eval_result.items():
        print(f"{mode:>10} | {k:>3} | {metrics['hit_rate']:>10.4f} | {metrics['mrr']:>10.4f}")

---
## 문제 7. FastAPI 벡터 검색 API 서버 + 캐시 + Rate Limiting (10점)

FastAPI 기반의 검색 API 서버를 구현하세요. Pydantic 모델, 인메모리 캐시, Rate Limiter를 포함합니다.

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import uvicorn
import threading
import time
from collections import defaultdict

# === Pydantic 모델 ===
# TODO: SearchRequest 정의 (query: str, top_k: int=5, mode: str="hybrid", filters: Optional[Dict])
class SearchRequest(BaseModel):
    pass  # TODO

# TODO: SearchResult 정의 (id: str, text: str, score: float, metadata: Dict)
class SearchResult(BaseModel):
    pass  # TODO

# TODO: SearchResponse 정의 (query: str, results: List[SearchResult], latency_ms: float, cached: bool)
class SearchResponse(BaseModel):
    pass  # TODO

# === 캐시 시스템 ===
class InMemoryCache:
    def __init__(self, ttl_seconds=60):
        self.cache = {}
        self.ttl = ttl_seconds
    
    def get(self, key):
        # TODO: TTL 체크 후 캐시 반환 (만료되었으면 삭제 후 None)
        pass
    
    def set(self, key, value):
        # TODO: 타임스탬프와 함께 저장
        pass
    
    def make_key(self, query, top_k, mode):
        # TODO: 검색 파라미터를 해시하여 캐시 키 생성
        pass
    
    @property
    def hit_count(self):
        # TODO
        pass

# === Rate Limiter ===
class RateLimiter:
    def __init__(self, max_requests=30, window_seconds=60):
        self.max_requests = max_requests
        self.window = window_seconds
        self.requests = defaultdict(list)
    
    def is_allowed(self, client_ip):
        # TODO: 현재 시간 기준 window 내 요청 수 체크
        # TODO: 초과 시 False, 아니면 기록 후 True
        pass

# === FastAPI 앱 ===
app = FastAPI(title="Hybrid Search API")
cache = InMemoryCache(ttl_seconds=60)
limiter = RateLimiter(max_requests=30)

@app.middleware("http")
async def rate_limit_middleware(request: Request, call_next):
    # TODO: client IP 추출 → rate limiter 체크 → 초과 시 429 반환
    pass

@app.post("/search", response_model=SearchResponse)
async def search_endpoint(request: SearchRequest):
    # TODO: 캐시 확인 → 미스 시 hybrid_search 호출 → 결과 캐시 저장 → 응답 반환
    pass

@app.get("/health")
async def health():
    # TODO: 상태 정보 반환 (document count, cache size, etc.)
    pass

# 서버 실행 (백그라운드)
def run():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run, daemon=True)
server_thread.start()
time.sleep(3)

# API 테스트
import httpx
resp = httpx.post("http://localhost:8000/search", json={"query": "컨테이너 배포", "top_k": 3})
print(f"Status: {resp.status_code}")
print(f"Results: {resp.json()}")

---
## 문제 8 (킬러). 자동 청킹 전략 최적화 시스템 (10점)

고정 크기, 문장 단위, 의미 기반 청킹 3가지 전략을 구현하고, 검색 품질(Hit Rate@3)로 최적 전략을 자동 선택하는 시스템을 만드세요.

In [ ]:
LONG_DOCUMENT = """
마이크로서비스 아키텍처는 대규모 애플리케이션을 작은 독립적인 서비스로 분해하는 설계 패턴입니다. 각 서비스는 특정 비즈니스 기능을 담당하며, 독립적으로 개발, 배포, 확장할 수 있습니다.

서비스 간 통신은 주로 REST API 또는 gRPC를 사용합니다. 비동기 통신이 필요한 경우 Kafka나 RabbitMQ 같은 메시지 브로커를 활용합니다. 이벤트 드리븐 아키텍처를 적용하면 서비스 간 결합도를 낮출 수 있습니다.

컨테이너화는 마이크로서비스 배포의 핵심입니다. Docker로 각 서비스를 패키징하고, 쿠버네티스로 오케스트레이션합니다. 서비스 메시(Service Mesh)인 Istio를 사용하면 트래픽 관리, 보안, 관찰 가능성을 투명하게 제공할 수 있습니다.

데이터 관리에서는 각 서비스가 자체 데이터베이스를 소유하는 Database per Service 패턴을 권장합니다. 분산 트랜잭션이 필요한 경우 Saga 패턴을 사용합니다. CQRS(Command Query Responsibility Segregation) 패턴으로 읽기와 쓰기를 분리하면 성능을 최적화할 수 있습니다.

모니터링과 로깅은 분산 시스템에서 필수적입니다. ELK 스택(Elasticsearch, Logstash, Kibana)으로 중앙 집중식 로깅을 구현하고, Prometheus와 Grafana로 메트릭을 수집하고 시각화합니다. 분산 추적은 Jaeger나 Zipkin을 사용하여 서비스 간 요청 흐름을 추적합니다.

보안은 API 게이트웨이에서 인증/인가를 중앙 처리하고, 서비스 간 통신은 mTLS로 암호화합니다. OAuth2와 JWT를 활용한 토큰 기반 인증이 일반적입니다. 서비스별 네트워크 정책으로 최소 권한 원칙을 적용합니다.
"""

EVAL_QUERIES = [
    ("마이크로서비스 간 통신 방법은?", "서비스 간 통신은 주로 REST API 또는 gRPC를 사용"),
    ("컨테이너 오케스트레이션 도구는?", "쿠버네티스로 오케스트레이션"),
    ("분산 트랜잭션 처리 패턴은?", "Saga 패턴"),
    ("로깅 시스템 구성은?", "ELK 스택"),
    ("서비스 간 보안 통신은?", "mTLS로 암호화"),
]

def chunk_fixed_size(text, size=150, overlap=30):
    """고정 크기 청킹"""
    # TODO
    pass

def chunk_by_sentence(text):
    """문장 단위 청킹 (마침표 기준, 2-3문장씩 그룹)"""
    # TODO: 마침표 기준 문장 분리 → 2-3문장씩 묶어서 청크 생성
    pass

def chunk_semantic(text, model, threshold=0.5):
    """
    의미 기반 청킹: 연속 문장 간 임베딩 유사도가 threshold 이하로 떨어지는 지점에서 분할
    """
    # TODO: 문장 분리
    # TODO: 각 문장 임베딩 생성
    # TODO: 연속 문장 간 코사인 유사도 계산
    # TODO: 유사도가 threshold 이하인 지점에서 분할
    # TODO: 분할된 구간의 문장들을 합쳐서 청크 생성
    pass

def evaluate_chunking_strategy(chunks, eval_queries, collection_name):
    """청킹 전략의 검색 품질 평가 (Hit Rate@3)"""
    client = chromadb.Client()
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # TODO: 컨렉션 생성 + 청크 적재
    # TODO: 각 평가 쿼리로 검색 (top 3)
    # TODO: 검색 결과에 정답 키워드가 포함되면 hit
    # TODO: Hit Rate 반환
    pass

# 3가지 전략 비교
model = SentenceTransformer('all-MiniLM-L6-v2')
strategies = {
    "fixed_size": chunk_fixed_size(LONG_DOCUMENT),
    "sentence": chunk_by_sentence(LONG_DOCUMENT),
    "semantic": chunk_semantic(LONG_DOCUMENT, model, threshold=0.5),
}

print("=" * 50)
print(f"{'\uc804\ub7b5':>12} | {'\uccad\ud06c \uc218':>7} | {'Hit Rate@3':>11}")
print("-" * 50)
for name, chunks in strategies.items():
    hr = evaluate_chunking_strategy(chunks, EVAL_QUERIES, f"eval_{name}")
    print(f"{name:>12} | {len(chunks):>7} | {hr:>11.4f}")

# TODO: 최적 전략 자동 선택
best = None  # TODO
print(f"\n\U0001f3c6 최적 전략: {best}")

---
## 문제 9 (킬러). 비동기 배치 임베딩 + 동기 대비 성능 비교 (10점)

100개 문서를 배치 단위로 동기/비동기 임베딩 처리하고 성능을 비교하세요. 재시도 로직과 진행률 표시를 포함합니다.

In [ ]:
import asyncio

# 100개 테스트 문서 생성
BATCH_DOCS = [f"테스트 문서 {i}: {'AI \uae30\uc220' if i%3==0 else '\ud074\ub77c\uc6b0\ub4dc \uc778\ud504\ub77c' if i%3==1 else '\ub370\uc774\ud130 \ubd84\uc11d'} 관련 내용입니다. 문서 번호는 {i}번입니다." for i in range(100)]

class BatchEmbeddingProcessor:
    def __init__(self, model_name='all-MiniLM-L6-v2', batch_size=10, max_retries=2):
        self.model = SentenceTransformer(model_name)
        self.batch_size = batch_size
        self.max_retries = max_retries
        self.processed = 0
        self.failed_batches = []
    
    def _split_batches(self, documents):
        """문서 리스트를 배치로 분할"""
        # TODO: batch_size 단위로 분할하여 list of lists 반환
        pass
    
    def process_sync(self, documents):
        """동기 처리: 순차적으로 배치 임베딩"""
        batches = self._split_batches(documents)
        all_embeddings = []
        start = time.time()
        
        for i, batch in enumerate(batches):
            # TODO: 배치 임베딩 생성
            # TODO: 실패 시 max_retries까지 재시도
            # TODO: 진행률 출력 (매 배치마다)
            pass
        
        elapsed = time.time() - start
        return np.vstack(all_embeddings), elapsed
    
    async def _embed_batch_async(self, batch, batch_idx):
        """단일 배치 비동기 임베딩 (실제로는 스레드풀에서 실행)"""
        loop = asyncio.get_event_loop()
        for attempt in range(self.max_retries + 1):
            try:
                # TODO: run_in_executor로 CPU 바운드 작업 비동기 실행
                embeddings = None
                self.processed += len(batch)
                return embeddings
            except Exception as e:
                if attempt == self.max_retries:
                    self.failed_batches.append(batch_idx)
                    return None
    
    async def process_async(self, documents):
        """비동기 처리: 동시에 여러 배치 처리"""
        batches = self._split_batches(documents)
        start = time.time()
        self.processed = 0
        self.failed_batches = []
        
        # TODO: 모든 배치에 대해 비동기 태스크 생성
        # TODO: asyncio.gather로 동시 실행
        # TODO: None인 결과(실패 배치) 필터링
        # TODO: 진행률 표시
        
        elapsed = time.time() - start
        all_embeddings = None  # TODO: 성공한 결과 합치기
        return all_embeddings, elapsed

processor = BatchEmbeddingProcessor(batch_size=10)

# 동기 처리
sync_embeddings, sync_time = processor.process_sync(BATCH_DOCS)
print(f"\n[동기] {len(BATCH_DOCS)}개 문서, 소요시간: {sync_time:.2f}초")

# 비동기 처리
async_embeddings, async_time = asyncio.get_event_loop().run_until_complete(
    processor.process_async(BATCH_DOCS)
)
print(f"[비동기] {len(BATCH_DOCS)}개 문서, 소요시간: {async_time:.2f}초")
print(f"\n속도 비교: 비동기가 {sync_time/async_time:.1f}배 {'\ube60\ub984' if async_time < sync_time else '\ub290\ub9bc'}")
print(f"실패 배치: {len(processor.failed_batches)}개")

# 임베딩 결과 검증 (두 방식의 결과가 동일한지)
# TODO: 코사인 유사도로 동기/비동기 결과 일치 확인

---
## 문제 10 (킬러). Production RAG 평가 파이프라인 통합 (10점)

RAG 시스템의 전체 파이프라인(검색 → 생성 → 평가)을 구현하세요.
MRR, ROUGE-L, Faithfulness 평가를 포함합니다.

In [ ]:
RAG_DOCUMENTS = [
    "쿠버네티스(K8s)는 컨테이너 오케스트레이션 플랫폼으로 자동 스케일링, 셀프 힐링, 롤링 업데이트를 지원합니다.",
    "Docker Compose는 멀티 컨테이너 앱을 YAML로 정의하고 단일 명령으로 실행하는 도구입니다.",
    "Terraform은 HashiCorp의 IaC 도구로 AWS, GCP, Azure 등 멀티클라우드 인프라를 코드로 관리합니다.",
    "Redis Sentinel은 마스터-슬레이브 구조에서 자동 장애 복구(failover)를 제공합니다.",
    "Kafka Connect는 데이터베이스, 파일시스템 등 외부 시스템과 Kafka 간의 데이터 이동을 자동화합니다.",
    "Prometheus의 PromQL은 시계열 데이터를 쿼리하는 함수형 언어로 rate, histogram_quantile 등의 함수를 제공합니다.",
    "Istio 서비스 메시는 사이드카 프록시(Envoy)를 통해 트래픽 관리, mTLS, 관찰 가능성을 투명하게 제공합니다.",
    "FastAPI의 Depends()는 의존성 주입 시스템으로 인증, DB 세션, 캐시 등을 엔드포인트에 자동 주입합니다.",
]

RAG_EVAL = [
    {"question": "쿠버네티스의 자동 복구 기능은 무엇인가요?", "reference": "쿠버네티스는 셀프 힐링 기능으로 장애가 발생한 파드를 자동으로 재시작합니다.", "relevant_docs": [0]},
    {"question": "여러 클라우드에 인프라를 배포하는 도구는?", "reference": "Terraform은 AWS, GCP, Azure 등 멀티클라우드 인프라를 코드로 관리하는 IaC 도구입니다.", "relevant_docs": [2]},
    {"question": "Kafka와 외부 시스템을 연결하는 방법은?", "reference": "Kafka Connect를 사용하여 데이터베이스 등 외부 시스템과의 데이터 이동을 자동화할 수 있습니다.", "relevant_docs": [4]},
    {"question": "서비스 간 보안 통신을 투명하게 처리하는 방법은?", "reference": "Istio 서비스 메시의 사이드카 프록시를 통해 mTLS 기반의 암호화된 통신을 투명하게 제공합니다.", "relevant_docs": [6]},
    {"question": "FastAPI에서 인증 로직을 재사용하는 방법은?", "reference": "FastAPI의 Depends() 의존성 주입 시스템을 통해 인증 로직을 엔드포인트에 자동 주입할 수 있습니다.", "relevant_docs": [7]},
]

class RAGEvaluationPipeline:
    def __init__(self, documents):
        self.documents = documents
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.openai_client = OpenAI()
        
        # TODO: ChromaDB에 문서 적재
        self.client = chromadb.Client()
        self.collection = None  # TODO
    
    def retrieve(self, question, top_k=3):
        """문서 검색"""
        # TODO: ChromaDB 검색, 문서 텍스트와 인덱스 반환
        pass
    
    def generate_answer(self, question, context_docs):
        """검색된 문서를 기반으로 LLM 답변 생성"""
        context = "\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(context_docs)])
        prompt = f"""다음 참고 문서만을 기반으로 질문에 답하세요. 참고 문서에 없는 내용은 답하지 마세요.

[참고 문서]
{context}

[질문] {question}

[답변]"""
        # TODO: LLM 호출
        pass
    
    def evaluate_retrieval(self, eval_data, top_k=3):
        """검색 품질 평가 (MRR@K)"""
        # TODO: 각 질문에 대해 검색 실행
        # TODO: relevant_docs와 비교하여 MRR 계산
        pass
    
    def evaluate_answer_rouge(self, generated, reference):
        """ROUGE-L F1 계산"""
        # TODO: 직접 구현 또는 rouge_scorer 사용
        pass
    
    def evaluate_faithfulness(self, question, answer, context_docs):
        """
        Faithfulness 평가: 답변의 각 claim이 검색 문서에 근거하는지 LLM으로 검증
        
        Returns:
            dict: {"faithful": bool, "score": float (0-1), "reason": str}
        """
        prompt = f"""다음 답변이 제공된 참고 문서에만 근거하는지 평가하세요.

[참고 문서]
{chr(10).join(context_docs)}

[답변]
{answer}

답변의 모든 주장이 참고 문서에 근거하면 faithful=true, 문서에 없는 정보가 포함되면 faithful=false입니다.
JSON으로만 응답하세요:
{{"faithful": true/false, "score": 0.0~1.0, "reason": "한줄 설명"}}"""
        # TODO: LLM 호출 및 JSON 파싱
        pass
    
    def run_full_evaluation(self, eval_data):
        """전체 RAG 평가 실행"""
        results = []
        
        for item in eval_data:
            q = item["question"]
            ref = item["reference"]
            
            # TODO: 검색
            retrieved_docs = None
            
            # TODO: 답변 생성
            answer = None
            
            # TODO: ROUGE-L 평가
            rouge = None
            
            # TODO: Faithfulness 평가
            faith = None
            
            results.append({
                "question": q,
                "answer": answer,
                "rouge_l": rouge,
                "faithful": faith["faithful"] if faith else None,
                "faith_score": faith["score"] if faith else None,
            })
        
        # TODO: 검색 MRR 계산
        mrr = self.evaluate_retrieval(eval_data)
        
        # TODO: 종합 리포트 출력
        print("=" * 65)
        print("RAG 평가 종합 리포트")
        print("=" * 65)
        print(f"  검색 품질 (MRR@3): {mrr:.4f}")
        print(f"  평균 ROUGE-L: {np.mean([r['rouge_l'] for r in results]):.4f}")
        print(f"  Faithfulness 비율: {sum(1 for r in results if r['faithful'])/len(results):.0%}")
        print(f"  평균 Faith Score: {np.mean([r['faith_score'] for r in results if r['faith_score']]):.4f}")
        print("-" * 65)
        for r in results:
            print(f"  Q: {r['question'][:40]}...")
            print(f"    ROUGE: {r['rouge_l']:.4f} | Faithful: {r['faithful']} ({r['faith_score']:.2f})")
        
        return results

pipeline = RAGEvaluationPipeline(RAG_DOCUMENTS)
results = pipeline.run_full_evaluation(RAG_EVAL)